# Computational Time Experiments

This notebook benchmarks FA and ZIFA runtime under varying cell counts and gene counts.

Workflow:
1. Read the raw 10x mouse brain h5 data from `datasets/mouse_brain/datasets/1M_neurons_filtered_gene_bc_matrices_h5.h5`.
2. Preprocess/sample one shared 30k-cell x 2,000-gene log1p matrix from the raw data.
3. Run four experiment groups in order: FA 1,000 genes, FA 2,000 genes, ZIFA 1,000 genes, ZIFA 2,000 genes.
4. Within each group, CPU and GPU jobs can run in parallel.
5. Save runtime tables/logs to `../results/scPI`; display figures only.

The executable preprocess cell below now requires the raw mouse brain h5 file; it does not use pre-sampled `.h5ad` files.

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd()
DATA_DIR = Path("datasets/mouse_brain/datasets")
CORTEX_DIR = Path("datasets/cortex/datasets")
RESULT_DIR = Path("../results/scPI")
SAMPLED_DIR = RESULT_DIR / "sampled_data"
JOB_DIR = RESULT_DIR / "runtime_jobs"
LOG_DIR = RESULT_DIR / "logs"

for path in [RESULT_DIR, SAMPLED_DIR, JOB_DIR, LOG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

RAW_MOUSE_BRAIN = DATA_DIR / "1M_neurons_filtered_gene_bc_matrices_h5.h5"
BASE_MATRIX = SAMPLED_DIR / "mouse_brain_30k_2000g_log1p.npy"
SAMPLE_META = SAMPLED_DIR / "mouse_brain_30k_2000g_meta.json"

SEED = 1234
K = 10
CELL_SIZES = [5000, 10000, 15000, 20000, 30000]
GENE_SIZES = [1000, 2000]

# Runtime settings. Adjust these on the target server before running.
MAX_CPU_JOBS = 1
GPU_IDS = [0]  # e.g. [0, 1, 2, 3] on a multi-GPU server. Use [] to force VI/AVI onto CPU.
OMP_NUM_THREADS_PER_CPU_JOB = "16"

# Match the original timing notebooks unless you intentionally tune these.
VI_THRESHOLD = 1e-2
VI_BATCH_SIZE = 128
VI_LR = 5e-4
VI_MAX_EPOCHS = 100000

print("Project:", PROJECT_DIR)
print("Raw mouse brain exists:", RAW_MOUSE_BRAIN.exists(), RAW_MOUSE_BRAIN)
print("Results:", RESULT_DIR)

## Raw Data Download Location

The mouse brain raw data is from 10x Genomics:

`https://cf.10xgenomics.com/samples/cell-exp/1.3.0/1M_neurons/1M_neurons_filtered_gene_bc_matrices_h5.h5`

Downloaded local path:

`datasets/mouse_brain/datasets/1M_neurons_filtered_gene_bc_matrices_h5.h5`

## Reference Raw Preprocess Code

The block below records the original raw-read + preprocess logic from the old notebook. It is intentionally commented for provenance. The executable preprocess cell after it performs the same raw read + sample idea, but caches one shared 30k x 2,000 matrix instead of saving every scale.

In [ ]:
# Original-style raw preprocess code, kept for reference only.
#
# import scanpy as sc
# import numpy as np
# import os
#
# adata = sc.read_10x_h5("datasets/mouse_brain/datasets/1M_neurons_filtered_gene_bc_matrices_h5.h5")
# adata.var_names_make_unique()
#
# cell_sizes = [5000, 10000, 15000, 20000, 30000]
# gene_sizes = [1000, 2000]
#
# for n_cells in cell_sizes:
#     cell_idx = np.random.choice(adata.n_obs, size=n_cells, replace=False)
#     adata_cell_sub = adata[cell_idx, :].copy()
#
#     expression = adata_cell_sub.X.toarray() if hasattr(adata_cell_sub.X, "toarray") else adata_cell_sub.X
#     gene_std = np.std(expression, axis=0)
#     sorted_gene_idx = np.argsort(gene_std)[::-1]
#
#     for n_genes in gene_sizes:
#         selected_genes = sorted_gene_idx[:n_genes]
#         adata_sub = adata_cell_sub[:, selected_genes].copy()
#         filename = f"datasets/mouse_brain/datasets/neurons_{n_cells//1000}k_{n_genes}g.h5ad"
#         adata_sub.write(filename)
#
# Cortex raw preprocess reference:
#
# import pandas as pd
# from sklearn.model_selection import train_test_split
#
# data_path = "datasets/cortex/datasets/"
# X = pd.read_csv(data_path + "expression_mRNA_17-Aug-2014.txt", sep="\t", low_memory=False).T
# clusters = np.array(X[7], dtype=str)[2:]
# celltypes, labels = np.unique(clusters, return_inverse=True)
# gene_names = np.array(X.iloc[0], dtype=str)[10:]
# X = X.loc[:, 10:]
# X = X.drop(X.index[0])
# expression = np.array(X, dtype=int)[1:]
# selected = np.std(expression, axis=0).argsort()[-558:][::-1]
# expression = expression[:, selected]
# X_train, X_test, c_train, c_test = train_test_split(expression, labels, random_state=0)

## Preprocess and Sample One Shared Dataset

This cell reads the raw 10x mouse brain h5, samples 30k cells once, ranks genes by standard deviation on those sampled cells, keeps the top 2,000 genes, applies `log1p`, and saves one shared base matrix. All four runtime panels slice from this same raw-derived matrix, so the data are identical across FA/ZIFA and 1,000/2,000-gene settings.

In [ ]:
def ensure_base_sample(force: bool = False) -> np.ndarray:
    if BASE_MATRIX.exists() and SAMPLE_META.exists() and not force:
        print(f"Using cached raw-derived base matrix: {BASE_MATRIX}")
        return np.load(BASE_MATRIX, mmap_mode="r")

    if not RAW_MOUSE_BRAIN.exists():
        raise FileNotFoundError(
            f"Raw mouse brain h5 is required for this notebook: {RAW_MOUSE_BRAIN}. "
            "Download it first, then rerun this cell."
        )

    import scanpy as sc
    from scipy import sparse

    rng = np.random.default_rng(SEED)
    print(f"Reading raw 10x data: {RAW_MOUSE_BRAIN}")
    adata = sc.read_10x_h5(RAW_MOUSE_BRAIN)
    adata.var_names_make_unique()

    max_cells = max(CELL_SIZES)
    max_genes = max(GENE_SIZES)
    if adata.n_obs < max_cells:
        raise ValueError(f"Raw data has only {adata.n_obs} cells, but {max_cells} are required.")

    cell_idx = rng.choice(adata.n_obs, size=max_cells, replace=False)
    adata_cell = adata[cell_idx, :].copy()
    X_sparse_or_dense = adata_cell.X

    print("Computing gene standard deviations on the shared 30k-cell sample...")
    if sparse.issparse(X_sparse_or_dense):
        mean = np.asarray(X_sparse_or_dense.mean(axis=0)).ravel()
        mean_sq = np.asarray(X_sparse_or_dense.power(2).mean(axis=0)).ravel()
        gene_std = np.sqrt(np.maximum(mean_sq - mean ** 2, 0))
    else:
        gene_std = np.std(np.asarray(X_sparse_or_dense), axis=0)

    gene_idx = np.argsort(gene_std)[::-1][:max_genes]
    adata_base = adata_cell[:, gene_idx].copy()
    X = adata_base.X.toarray() if hasattr(adata_base.X, "toarray") else np.asarray(adata_base.X)
    X = np.log1p(X).astype(np.float32, copy=False)

    np.save(BASE_MATRIX, X)
    np.save(SAMPLED_DIR / "mouse_brain_30k_cell_idx.npy", cell_idx)
    np.save(SAMPLED_DIR / "mouse_brain_top2000_gene_idx.npy", gene_idx)
    SAMPLE_META.write_text(json.dumps({
        "source": str(RAW_MOUSE_BRAIN),
        "seed": SEED,
        "n_cells": int(X.shape[0]),
        "n_genes": int(X.shape[1]),
        "cell_sizes": CELL_SIZES,
        "gene_sizes": GENE_SIZES,
        "transform": "log1p",
        "sampling": "One 30k-cell sample and one top-2000-gene ranking shared by all panels.",
    }, indent=2))
    print(f"Saved shared base matrix: {BASE_MATRIX}, shape={X.shape}")
    return np.load(BASE_MATRIX, mmap_mode="r")

X_base = ensure_base_sample(force=False)
print("Base matrix shape:", X_base.shape)

## Job Runner

The notebook writes a small local runner so each timing task executes in a fresh Python process. This avoids CUDA state conflicts and lets CPU and GPU jobs run in parallel inside each experiment group.

In [ ]:
RUNNER_PATH = RESULT_DIR / "runtime_job_runner.py"
RUNNER_PATH.write_text('\nfrom __future__ import annotations\n\nimport json\nimport os\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\n\n\ndef main(job_json: str) -> None:\n    spec_path = Path(job_json)\n    spec = json.loads(spec_path.read_text())\n\n    os.environ.setdefault("OMP_NUM_THREADS", str(spec.get("omp_num_threads", 1)))\n    os.environ.setdefault("MKL_NUM_THREADS", str(spec.get("omp_num_threads", 1)))\n    os.environ.setdefault("OPENBLAS_NUM_THREADS", str(spec.get("omp_num_threads", 1)))\n\n    import warnings\n    warnings.filterwarnings("ignore")\n\n    X_base = np.load(spec["base_matrix"], mmap_mode="r")\n    n_cells = int(spec["n_cells"])\n    n_genes = int(spec["n_genes"])\n    X = np.asarray(X_base[:n_cells, :n_genes], dtype=np.float32)\n\n    result_path = Path(spec["result_path"])\n    result_path.parent.mkdir(parents=True, exist_ok=True)\n\n    start = time.time()\n    method = spec["method"]\n    K = int(spec["K"])\n    seed = int(spec.get("seed", 1234))\n    np.random.seed(seed)\n\n    if method == "FA":\n        from sklearn.decomposition import FactorAnalysis\n        model = FactorAnalysis(n_components=K, random_state=seed)\n        model.fit(X)\n        _ = model.transform(X)\n        extra = {"backend": "sklearn.FactorAnalysis", "device": "cpu"}\n\n    elif method in {"FA_VI", "FA_AVI"}:\n        import pyro\n        import torch\n        from FA import FA\n        pyro.clear_param_store()\n        torch.manual_seed(seed)\n        inference = "vi" if method == "FA_VI" else "amortized"\n        model = FA(\n            K=K,\n            method=inference,\n            threshold=float(spec["threshold"]),\n            batch_size=int(spec["batch_size"]),\n            lr=float(spec["lr"]),\n            max_epochs=int(spec["max_epochs"]),\n        )\n        model.fit(X)\n        extra = {"backend": "FA.py", "device": "cuda" if torch.cuda.is_available() else "cpu"}\n\n    elif method == "block_ZIFA":\n        from ZIFA import ZIFA\n        nonzero_cols = np.sum(X, axis=0) > 0\n        X_use = X[:, nonzero_cols]\n        model = ZIFA(K=K, method="block")\n        model.fit(X_use)\n        extra = {"backend": "ZIFA.py:block", "device": "cpu", "n_genes_after_filter": int(X_use.shape[1])}\n\n    elif method in {"ZIFA_VI", "ZIFA_AVI"}:\n        import pyro\n        import torch\n        from ZIFA import ZIFA\n        pyro.clear_param_store()\n        torch.manual_seed(seed)\n        inference = "vi" if method == "ZIFA_VI" else "amortized"\n        model = ZIFA(\n            K=K,\n            method="pyro",\n            inference=inference,\n            threshold=float(spec["threshold"]),\n            batch_size=int(spec["batch_size"]),\n            lr=float(spec["lr"]),\n            max_epochs=int(spec["max_epochs"]),\n        )\n        model.fit(X)\n        extra = {"backend": "ZIFA.py:pyro", "device": "cuda" if torch.cuda.is_available() else "cpu"}\n\n    else:\n        raise ValueError(f"Unknown method: {method}")\n\n    elapsed = time.time() - start\n    payload = {\n        "panel": spec["panel"],\n        "method": method,\n        "label": spec["label"],\n        "n_cells": n_cells,\n        "n_genes": n_genes,\n        "time_sec": elapsed,\n        "time_min": elapsed / 60.0,\n        "seed": seed,\n        **extra,\n    }\n    result_path.write_text(json.dumps(payload, indent=2))\n    print(json.dumps(payload, indent=2))\n\n\nif __name__ == "__main__":\n    main(sys.argv[1])\n')
print("Wrote runner:", RUNNER_PATH)

## Runtime Helpers

In [ ]:
def result_json_path(panel: str, method: str, n_cells: int, n_genes: int) -> Path:
    return JOB_DIR / f"{panel}_{method}_{n_cells}_{n_genes}.json"


def make_job(panel: str, n_genes: int, method: str, label: str, n_cells: int, resource: str) -> dict:
    result_path = result_json_path(panel, method, n_cells, n_genes)
    return {
        "panel": panel,
        "method": method,
        "label": label,
        "resource": resource,
        "n_cells": n_cells,
        "n_genes": n_genes,
        "K": K,
        "seed": SEED,
        "base_matrix": str(BASE_MATRIX),
        "result_path": str(result_path),
        "threshold": VI_THRESHOLD,
        "batch_size": VI_BATCH_SIZE,
        "lr": VI_LR,
        "max_epochs": VI_MAX_EPOCHS,
        "omp_num_threads": OMP_NUM_THREADS_PER_CPU_JOB,
    }


def collect_results() -> pd.DataFrame:
    rows = []
    for p in sorted(JOB_DIR.glob("*.json")):
        try:
            rows.append(json.loads(p.read_text()))
        except json.JSONDecodeError:
            print("Skipping incomplete json:", p)
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(["panel", "method", "n_genes", "n_cells"])
        df.to_csv(RESULT_DIR / "runtime_results.csv", index=False)
    return df


def run_group(panel: str, n_genes: int, methods: list[tuple[str, str, str]]) -> None:
    jobs = []
    for method, label, resource in methods:
        for n_cells in CELL_SIZES:
            jobs.append(make_job(panel, n_genes, method, label, n_cells, resource))

    pending = [job for job in jobs if not Path(job["result_path"]).exists()]
    skipped = len(jobs) - len(pending)
    print(f"{panel}: {len(pending)} pending, {skipped} already complete")
    if not pending:
        return

    running = []
    next_gpu = 0

    def launch(job: dict):
        nonlocal next_gpu
        env = os.environ.copy()
        env["PYTHONUNBUFFERED"] = "1"
        env["OMP_NUM_THREADS"] = str(job["omp_num_threads"])
        env["MKL_NUM_THREADS"] = str(job["omp_num_threads"])
        env["OPENBLAS_NUM_THREADS"] = str(job["omp_num_threads"])
        if job["resource"] == "gpu" and GPU_IDS:
            gpu = GPU_IDS[next_gpu % len(GPU_IDS)]
            next_gpu += 1
            env["CUDA_VISIBLE_DEVICES"] = str(gpu)
            slot = f"gpu{gpu}"
        elif job["resource"] == "gpu":
            env["CUDA_VISIBLE_DEVICES"] = ""
            slot = "cpu_forced_gpu_job"
        else:
            env["CUDA_VISIBLE_DEVICES"] = ""
            slot = "cpu"

        spec_path = JOB_DIR / f"spec_{job['panel']}_{job['method']}_{job['n_cells']}_{job['n_genes']}.json"
        spec_path.write_text(json.dumps(job, indent=2))
        log_path = LOG_DIR / f"{job['panel']}_{job['method']}_{job['n_cells']}_{job['n_genes']}.log"
        log_fh = open(log_path, "w")
        proc = subprocess.Popen(
            [sys.executable, str(RUNNER_PATH), str(spec_path)],
            cwd=str(PROJECT_DIR),
            env=env,
            stdout=log_fh,
            stderr=subprocess.STDOUT,
        )
        print(f"Launched {job['panel']} {job['method']} {job['n_cells']} cells {job['n_genes']} genes on {slot}; log={log_path}")
        return {"proc": proc, "job": job, "log_fh": log_fh, "slot": slot}

    while pending or running:
        cpu_running = sum(1 for r in running if r["job"]["resource"] == "cpu")
        gpu_running = sum(1 for r in running if r["job"]["resource"] == "gpu")
        gpu_capacity = max(1, len(GPU_IDS)) if GPU_IDS else 1

        launched_any = False
        for job in list(pending):
            if job["resource"] == "cpu" and cpu_running < MAX_CPU_JOBS:
                running.append(launch(job))
                pending.remove(job)
                cpu_running += 1
                launched_any = True
            elif job["resource"] == "gpu" and gpu_running < gpu_capacity:
                running.append(launch(job))
                pending.remove(job)
                gpu_running += 1
                launched_any = True

        time.sleep(10 if running else 1)

        for r in list(running):
            rc = r["proc"].poll()
            if rc is None:
                continue
            r["log_fh"].close()
            running.remove(r)
            job = r["job"]
            if rc != 0:
                failed_log = LOG_DIR / f"{job['panel']}_{job['method']}_{job['n_cells']}_{job['n_genes']}.log"
                raise RuntimeError(
                    f"Job failed rc={rc}: {job['panel']} {job['method']} {job['n_cells']} {job['n_genes']}. "
                    f"See {failed_log}"
                )
            print(f"Finished {job['panel']} {job['method']} {job['n_cells']} cells")

    df = collect_results()
    print(df.tail())

## Run Four Experiment Groups

The groups are intentionally run in this order: FA 1,000 genes, FA 2,000 genes, ZIFA 1,000 genes, ZIFA 2,000 genes.

In [ ]:
FA_METHODS = [
    ("FA", "FA", "cpu"),
    ("FA_VI", "VI", "gpu"),
    ("FA_AVI", "AVI", "gpu"),
]

ZIFA_METHODS = [
    ("block_ZIFA", "block_ZIFA", "cpu"),
    ("ZIFA_VI", "VI", "gpu"),
    ("ZIFA_AVI", "AVI", "gpu"),
]

run_group("FA_1000", 1000, FA_METHODS)
run_group("FA_2000", 2000, FA_METHODS)
run_group("ZIFA_1000", 1000, ZIFA_METHODS)
run_group("ZIFA_2000", 2000, ZIFA_METHODS)

runtime_df = collect_results()
runtime_df

## Plot Runtime Figure

Matplotlib default color cycle is used intentionally.

In [ ]:
runtime_df = collect_results()
if runtime_df.empty:
    raise ValueError("No runtime results found yet.")

panel_specs = [
    ("FA_1000", "FA - 1,000 genes", ["FA", "VI", "AVI"]),
    ("ZIFA_1000", "ZIFA - 1,000 genes", ["block_ZIFA", "VI", "AVI"]),
    ("FA_2000", "FA - 2,000 genes", ["FA", "VI", "AVI"]),
    ("ZIFA_2000", "ZIFA - 2,000 genes", ["block_ZIFA", "VI", "AVI"]),
]

fig, axes = plt.subplots(1, 4, figsize=(22, 4.8), sharey=False)
for ax, (panel, title, labels) in zip(axes, panel_specs):
    sub = runtime_df[runtime_df["panel"] == panel]
    for label in labels:
        s = sub[sub["label"] == label].sort_values("n_cells")
        ax.plot(s["n_cells"] / 1000, s["time_min"], marker="o", label=label)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Number of cells (thousands)", fontweight="bold")
    ax.set_ylabel("Computational time (min)", fontweight="bold")
    ax.grid(True, alpha=0.25)
    ax.legend(frameon=True)

fig.tight_layout()
plt.show()


In [ ]:
# Optional: display each panel separately.
for panel, title, labels in panel_specs:
    fig, ax = plt.subplots(figsize=(5.5, 4.8))
    sub = runtime_df[runtime_df["panel"] == panel]
    for label in labels:
        s = sub[sub["label"] == label].sort_values("n_cells")
        ax.plot(s["n_cells"] / 1000, s["time_min"], marker="o", label=label)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Number of cells (thousands)", fontweight="bold")
    ax.set_ylabel("Computational time (min)", fontweight="bold")
    ax.grid(True, alpha=0.25)
    ax.legend(frameon=True)
    fig.tight_layout()
    plt.show()
